# Using QuTiP with ParaQeet

In this example we demonstrate how QuTiP <cite data-cite="lambert2026qutip">(Lambert et al., 2026)</cite> objects could be used for modeling a quantum system, and combined with ParaQeet for propagation and optimization tasks. Here we follow the same example as before, of a state preparation task, but use QuTiP functions for modeling the system.

In [ ]:
import jax.numpy as jnp
import qutip as qt
import qutip_jax

import paraqeet as pq

qutip_jax.set_as_default()

Here we use the [qutip-jax](https://github.com/qutip/qutip-jax) package for better integration with ParaQeet, as ParaQeet deals with pure JAX objects.

## 1. Define the parameters, Hamiltonian function and gradient functions

Similar to the previous example, here we also define a `CosEnvelope` as our pulse ansatz. 

The Hamiltonian in this example is, similarly, given by
$$H(t) = \frac{\omega}{2} \sigma_z + u(A, \omega_d, t) \sigma_x, $$
with $u(A, \omega_d, t) = A\cos(\omega_d t)$.

In [ ]:
from paraqeet.signal.envelopes import Envelope

amplitude = pq.Quantity(
    value=jnp.array(1.55e8),
    min_value=jnp.array(0.0),
    max_value=jnp.array(5 * 1e8),
    unit="Hz",
    name="Amplitude",
    two_pi=True,
)

frequency = pq.Quantity(
    value=jnp.array(4.8e9 * 2 * jnp.pi),
    min_value=jnp.array(0.8 * 4.8e9 * 2 * jnp.pi),
    max_value=jnp.array(1.2 * 4.8e9 * 2 * jnp.pi),
    unit="Hz",
    name="lo_freq",
    two_pi=True,
)

t_final = 12e-9
t_final_qty = pq.Quantity(
    value=jnp.array(t_final),
    min_value=jnp.array(0.8 * t_final),
    max_value=jnp.array(1.2 * t_final),
    unit="s",
    name="t_final",
)

tls_freq = 4.8e9 * 2 * jnp.pi


class CosEnvelope(Envelope):
    """Cosine envelope."""

    def __init__(self, amplitude, frequency, t_final=None):
        super().__init__(amplitude, t_final)
        self.frequency = frequency

    def get_parameters(self):
        """Return parameters amplitude and frequency"""
        return [self.amplitude, self.frequency]

    def _evaluate(self, amp, freq, t: pq.Array):
        """Define a cosine envelope."""
        return amp * jnp.cos(freq * t)

    def get_value(self, times):
        """Return value of the Envelope."""
        amp = self.amplitude.get_value()
        freq = self.frequency.get_value()
        return self._evaluate(amp, freq, times)


cos_env = CosEnvelope(amplitude, frequency, t_final)


# We define a H(t) for a single time-point t using QuTiP and pass it to the qt_func_to_jax_func wrapper
# to convert into a JAX function which supports vectorized time-points as input.
def tls_hamiltonian(t: float):
    """Define a Two level system Hamiltonian."""
    return 0.5 * tls_freq * qt.sigmaz() + qt.sigmax() * cos_env.get_value(t)

In [ ]:
tls_hamiltonian(0.0)

We can see that the output of the tls_hamiltonian is a `Qobj`. For working with ParaQeet we convert the Hamiltonian function to a JAX-compatible function by using the wrapper function `qt_func_to_jax_func`.

In [ ]:
from paraqeet.hamiltonian.utils import qt_func_to_jax_func

jax_ham_func = qt_func_to_jax_func(tls_hamiltonian)
jax_ham_func(jnp.array([0.0, 1e-9]))

The `jax_ham_func` now works with a batch of time points and returns the Hamiltonian as a JAX `Array` for each time point.

Similar to the previous example, for cases where the gradients of the Hamiltonian are not required, e.g., for state propagation and gradient-free optimization, the `hamiltonian_and_gradient_func` can be defined by any function. Here we define a function that raises an exception when it is called.

In [ ]:
from paraqeet.eom.schroedinger_equation import SchroedingerEquation


def ham_grad(t):
    """Gradient of the Hamiltonian. Raises exception when called."""
    raise Exception("Hamiltonian gradients have not been defined.")


model = SchroedingerEquation(hamiltonian_func=jax_ham_func, hamiltonian_gradient_func=ham_grad)

## 2. Define propagation method and measurement function

Here we pick the standard `ExpmGOAT` method for propagation and `UnitaryFidelity` as our measurement function. We start from the identity matrix with the goal to prepare the Hadamard gate.

In [ ]:
from paraqeet.hamiltonian.utils import qobj_to_array
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelity
from paraqeet.measurement.utils import overlap_state_vector
from paraqeet.propagation.expm_goat import ExpmGOAT

init_qobj = qt.basis(2, 0)  # |0>
target_qobj = qt.basis(2, 1)  # |1>

# Convert Qobj to jax Array
init = qobj_to_array(init_qobj)
target = qobj_to_array(target_qobj)

prop = ExpmGOAT(eom_func=model.get_value, eom_gradient_func=model.get_gradient, resolution=100e9, initial_state=init)
times = jnp.array([0.0, t_final])

zeroone = StateTransferFidelity(
    propagation_func=prop.get_value,
    propagation_gradient_func=prop.get_gradient,
    target_state=target,
    overlap=overlap_state_vector,
)

In [ ]:
from plotting import plot_signal_and_dynamics

ts = jnp.linspace(0.0, t_final, 501)
plot_signal_and_dynamics(cos_env, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"])

In [ ]:
zeroone.get_value(times)

## 3. Gradient based optimization

For gradient-based optimization let's define the gradient of the Hamiltonian function also using QuTiP objects.

In [ ]:
def grad_tls_hamiltonian(t: pq.Array):
    """Value and gradient of the TLS Hamiltonian."""
    _, grads = cos_env.get_value_and_gradient(t)
    sigma_x = qobj_to_array(qt.sigmax())
    ham_grads = jnp.stack(
        [grads[:, 0].reshape((-1, 1, 1)) * sigma_x, grads[:, 1].reshape((-1, 1, 1)) * sigma_x], axis=1
    )
    return ham_grads

Finally we can redefine the model (and propagation and measurement function) with the updated `hamiltonian_and_gradient_func`.

In [ ]:
model = SchroedingerEquation(hamiltonian_func=jax_ham_func, hamiltonian_gradient_func=grad_tls_hamiltonian)

prop = ExpmGOAT(eom_func=model.get_value, eom_gradient_func=model.get_gradient, resolution=100e9, initial_state=init)
times = jnp.array([0.0, t_final])

zeroone = StateTransferFidelity(
    propagation_func=prop.get_value,
    propagation_gradient_func=prop.get_gradient,
    target_state=target,
    overlap=overlap_state_vector,
)

In [ ]:
zeroone.get_value(times)

In [ ]:
cos_env.get_parameters()

## 4. Create optmap and optimize

In [ ]:
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.scipy_optimizer_gradient import ScipyOptimizerGradient

optmap = OptimizationMap()
optmap.add(cos_env)
opt = ScipyOptimizerGradient(measure_and_gradient_func=zeroone.get_value_and_gradient, optimization_map=optmap)

In [ ]:
opt.optimize(times)

In [ ]:
plot_signal_and_dynamics(cos_env, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"])

In [ ]:
zeroone.get_value(times)

## References

- **(Lambert et al., 2026)** N. Lambert et al., "QuTiP 5: The quantum toolbox in Python," *Physics Reports* **1153**, 1–62 (2026).